# 02 — Stationarity Tests — HICP Eurozone

Formal determination of the integration order of the HICP series.

**Tests:** ADF (H0: unit root) | KPSS (H0: stationary) | Phillips-Perron (H0: unit root)

Applied to: level, diff(1), diff(12), diff(1)+diff(12).

**Input:** `data/processed/hicp_europe_index.parquet`  
**Determines:** d and D parameters for ARIMA/SARIMA

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron

ROOT = Path('..').resolve()         # tfg-forecasting/
MONOREPO = ROOT.parent              # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))
from shared.constants import DATE_TRAIN_END
plt.rcParams.update({"figure.figsize": (14, 4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
df = pd.read_parquet(ROOT / "data" / "processed" / "hicp_europe_index.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
train = df.loc[:DATE_TRAIN_END]
y = train["hicp_index"]
print(f"Train: {y.index.min().date()} -> {y.index.max().date()} ({len(y)} obs)")

## 1. Auxiliary Test Functions

In [ ]:
def run_adf(series, name=""):
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pval, lags, nobs = result[0], result[1], result[2], result[3]
    cv = result[4]
    reject = pval < 0.05
    return {"test": "ADF", "series": name, "statistic": round(stat,4), "p_value": round(pval,4),
            "lags": lags, "nobs": nobs, "cv_5%": round(cv["5%"],4),
            "reject_H0": reject, "conclusion": "Stationary" if reject else "Non-stationary"}

def run_kpss(series, name="", regression="c"):
    stat, pval, lags, cv = kpss(series.dropna(), regression=regression, nlags="auto")
    reject = pval < 0.05
    return {"test": f"KPSS({regression})", "series": name, "statistic": round(stat,4), "p_value": round(pval,4),
            "lags": lags, "cv_5%": round(cv["5%"],4),
            "reject_H0": reject, "conclusion": "Non-stationary" if reject else "Stationary"}

def run_pp(series, name=""):
    pp = PhillipsPerron(series.dropna())
    reject = pp.pvalue < 0.05
    return {"test": "Phillips-Perron", "series": name, "statistic": round(pp.stat,4), "p_value": round(pp.pvalue,4),
            "lags": pp.lags, "reject_H0": reject, "conclusion": "Stationary" if reject else "Non-stationary"}

def battery(series, name):
    return [run_adf(series, name), run_kpss(series, name, "c"), run_kpss(series, name, "ct"), run_pp(series, name)]

## 2. Tests on the Level Series

In [ ]:
results = battery(y, "HICP level")
pd.DataFrame(results)

## 3. First Difference (d=1)

In [ ]:
y_diff1 = y.diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y.index, y, linewidth=1)
axes[0].set_title("Level Series")
axes[1].plot(y_diff1.index, y_diff1, linewidth=1, color="#d62728")
axes[1].set_title("First Difference (d=1)")
axes[1].axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
results += battery(y_diff1, "HICP diff(1)")
pd.DataFrame(battery(y_diff1, "HICP diff(1)"))

## 4. Seasonal Difference (D=1, lag 12)

In [ ]:
y_diff12 = y.diff(12).dropna()
fig, ax = plt.subplots()
ax.plot(y_diff12.index, y_diff12, linewidth=1, color="#2ca02c")
ax.set_title("Seasonal Difference (lag 12)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
results += battery(y_diff12, "HICP diff(12)")
pd.DataFrame(battery(y_diff12, "HICP diff(12)"))

## 5. Double Difference: d=1 + D=1

In [ ]:
y_diff1_12 = y.diff().diff(12).dropna()
fig, ax = plt.subplots()
ax.plot(y_diff1_12.index, y_diff1_12, linewidth=1, color="#9467bd")
ax.set_title("Double Difference: diff(1) + diff(12)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
results += battery(y_diff1_12, "HICP diff(1)+diff(12)")
pd.DataFrame(battery(y_diff1_12, "HICP diff(1)+diff(12)"))

## 6. Summary Table

In [ ]:
summary = pd.DataFrame(results)
cols_show = ["test", "series", "statistic", "p_value", "conclusion"]
summary[cols_show].style.map(
    lambda v: "background-color: #d4edda" if v == "Stationary" else
              "background-color: #f8d7da" if v == "Non-stationary" else "",
    subset=["conclusion"]
)

## 7. Conclusion

**Joint interpretation ADF + KPSS + PP:**

| Transformation | ADF | KPSS | PP | Decision |
|---|---|---|---|---|
| Level | ? | ? | ? | |
| diff(1) | ? | ? | ? | |
| diff(12) | ? | ? | ? | |
| diff(1)+diff(12) | ? | ? | ? | |

*Fill with actual values after execution.*